# Comparing two revisions of a pipeline

A scenario with `revision=` runs the flow as it was at another commit, next to the current code.
`credit_pipeline.py`'s last commit added a flat $50 affordability fee to `ratio`; this compares it with the
commit before.

**How an experiment runs.** The runner feeds the *baseline* (the unchanged flow) and every *scenario* the
same records, one batch at a time. Each **metric** gets its own copy per scenario and collects through hooks:
a *global* metric reads the flow's output; a *step* metric (`nodes="<glob>"`) reads what matching steps
wrote, on the records that ran them. `summarise()` gives a typed summary; a **plot** draws one metric's
summaries across scenarios. `result.show()` lays it all out.

In [1]:
import polars as pl
from decider.experiments import *
from example_projects.experiments.credit_pipeline import build

ENTRY = "example_projects.experiments.credit_pipeline:build"
pipeline = build()
df = pl.DataFrame({
    "income": [1000.0, 2000.0, 1500.0, 800.0],
    "debt": [250.0, 500.0, 700.0, 250.0],
})
df

income,debt
f64,f64
1000.0,250.0
2000.0,500.0
1500.0,700.0
800.0,250.0


In [2]:
import subprocess
import example_projects.experiments.credit_pipeline as credit_pipeline

# The commit that added the fee, so this keeps working as the repository moves on.
fee = subprocess.run(["git", "log", "-1", "--format=%h", "--", credit_pipeline.__file__],
                     capture_output=True, text=True, check=True).stdout.strip()

result = (Experiment("version-drift", pipeline, df, entry=ENTRY)
          .scenario("before_fee", revision=f"{fee}^")
          .metric(Divergence(column="approved"),                                          # global
                  Divergence(name="ratio_changed", column="ratio", nodes="credit/ratio"),  # at a step
                  Histogram(name="ratio", column="ratio", nodes="credit/ratio", bins=8, range=(0.2, 0.6)))
          .plot(Changes(metric="divergence"), Distribution(metric="ratio"))
          .run())
result.plots["ratio"].show()

Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'start': 0.2, 'end': 0.25, 'records': 0}, {'scenario': 'baseline', 'start': 0.25, 'end': 0.3, 'records': 1}, {'scenario': 'baseline', 'start': 0.3, 'end': 0.35, 'records': 1}, {'scenario': 'baseline', 'start': 0.35, 'end': 0.4, 'records': 1}, {'scenario': 'baseline', 'start': 0.4, 'end': 0.44999999999999996, 'records': 0}, {'scenario': 'baseline', 'start': 0.44999999999999996, 'end': 0.5, 'records': 0}, {'scenario': 'baseline', 'start': 0.5, 'end': 0.55, 'records': 1}, {'scenario': 'baseline', 'start': 0.55, 'end': 0.6, 'records': 0}, {'scenario': 'before_fee', 'start': 0.2, 'end': 0.25, 'records': 0}, {'scenario': 'before_fee', 'start': 0.25, 'end': 0.3, 'records': 2}, {'scenario': 'before_fee', 'start': 0.3, 'end': 0.35, 'records': 1}, {'scenario': 'before_fee', 'start': 0.35, 'end': 0.4, 'records': 0}, {'scenario': 'before_fee', 'start': 0.4, 'end': 0.44999999999999996, 'records': 0}, {'scenario': 'before_fee', 'start': 0.44999999999999996, 'end': 0.5, 'records': 1}, {'scenario': 'before_fee', 'start': 0.5, 'end': 0.55, 'records': 0}, {'scenario': 'before_fee', 'start': 0.55, 'end': 0.6, 'records': 0}]}, 'mark': {'type': 'bar'}, 'encoding': {'color': {'field': 'scenario', 'type': 'nominal'}, 'opacity': {'condition': {'param': 'param_366548e304bb77de', 'value': 0.55}, 'value': 0.05}, 'tooltip': [{'field': 'scenario', 'type': 'nominal'}, {'field': 'start', 'format': '.4g', 'type': 'quantitative'}, {'field': 'end', 'format': '.4g', 'type': 'quantitative'}, {'field': 'records', 'type': 'quantitative'}], 'x': {'bin': 'binned', 'field': 'start', 'title': 'ratio', 'type': 'quantitative'}, 'x2': {'field': 'end'}, 'y': {'field': 'records', 'stack': None, 'type': 'quantitative'}}, 'params': [{'name': 'param_366548e304bb77de', 'select': {'type': 'point', 'fields': ['scenario']}, 'bind': 'legend'}], 'title': 'ratio by scenario', '$schema': 'https://vega.github.io/schema/vega-lite/v6.4.1.json'})

`ratio` is not in the flow's output (`approved` reads it), so the metrics on it attach to the step
`credit/ratio`. The fee moves every ratio but flips one approval. The other revision runs in a fresh
subprocess, because the live module is already imported under the same name.

In [3]:
result.show()